# Generation Experiments

This notebook prepares generation experiments by optimally matching frame-aligned article pairs by body length and sampling matched pairs, and runs the generation experiments.


## Section 0: Setup

### Imports

In [ ]:
import itertools
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, List

import anthropic
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm


from openai import OpenAI
import anthropic
from google import genai
from google.genai import types as genai_types


from news_agent_utils.events import GENERATION_QUESTIONS_BY_EVENT
from news_agent_utils.io import read_table, save_table, stable_hash
from news_agent_utils.llm import (
    create_provider_client,
    load_provider_api_key,
    run_cached_text_generation_call,
)

pd.set_option("display.max_colwidth", 160)


### Directories

In [ ]:
BASE_DIR = Path(".")
LLM_CACHE_DIR = BASE_DIR / "cache" / "generation" / "llm"
DATA_DIR = BASE_DIR / "data"

for path in [LLM_CACHE_DIR, DATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Experiment directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"LLM cache directory: {LLM_CACHE_DIR}")


### API Keys


In [ ]:
OPENAI_API_KEY = load_provider_api_key("openai")
ANTHROPIC_API_KEY = load_provider_api_key("anthropic")
GEMINI_API_KEY = load_provider_api_key("gemini")

OPENAI_CLIENT = create_provider_client("openai", OPENAI_API_KEY, openai_client_cls=OpenAI)
ANTHROPIC_CLIENT = create_provider_client("anthropic", ANTHROPIC_API_KEY, anthropic_module=anthropic)
GEMINI_CLIENT = create_provider_client("gemini", GEMINI_API_KEY, genai_module=genai)
PROVIDER_CLIENTS = {
    "openai": OPENAI_CLIENT,
    "anthropic": ANTHROPIC_CLIENT,
    "gemini": GEMINI_CLIENT,
}

print(f"Loaded OpenAI API key: {OPENAI_API_KEY is not None}.")
print(f"Loaded Anthropic API key: {ANTHROPIC_API_KEY is not None}.")
print(f"Loaded Gemini API key: {GEMINI_API_KEY is not None}.")
if not OPENAI_API_KEY:
    print("Set OPENAI_API_KEY before running OpenAI generation experiments.")
if not ANTHROPIC_API_KEY:
    print("Set ANTHROPIC_API_KEY before running Anthropic generation experiments.")
if not GEMINI_API_KEY:
    print("Set GEMINI_API_KEY before running Gemini generation experiments.")
if ANTHROPIC_API_KEY and anthropic is None:
    print("Anthropic key found, but the anthropic package is not importable.")
if GEMINI_API_KEY and genai is None:
    print("Gemini key found, but the google-genai package is not importable.")


## Section 1: Load and Join Article Framings

We use the dataframe with the articles and the dominant framing annotations, and we add extra checks to ensure that we don't have duplicate or missing articles. 

In [ ]:
ARTICLE_TABLE_NAME = "df_articles"
ANNOTATIONS_TABLE_NAME = "df_articles_with_dominant_framing_annotations"

df_articles = read_table(DATA_DIR, ARTICLE_TABLE_NAME)
df_dominant_framing_annotations = read_table(DATA_DIR, ANNOTATIONS_TABLE_NAME)

print(f"Loaded canonical articles: {len(df_articles)} row(s) from DATA_DIR / {ARTICLE_TABLE_NAME!r}.")
print(f"Loaded dominant framing annotations: {len(df_dominant_framing_annotations)} row(s) from DATA_DIR / {ANNOTATIONS_TABLE_NAME!r}.")

required_article_columns = {"article_id", "event", "slant", "title", "body", "url", "source"}
required_annotation_columns = {"article_id", "frame_1_score", "frame_2_score"}
missing_article_columns = required_article_columns - set(df_articles.columns)
missing_annotation_columns = required_annotation_columns - set(df_dominant_framing_annotations.columns)
assert not missing_article_columns, f"Canonical article table is missing columns: {sorted(missing_article_columns)}"
assert not missing_annotation_columns, f"Dominant framing annotations are missing columns: {sorted(missing_annotation_columns)}"


In [ ]:
article_duplicate_ids = (
    df_articles.loc[df_articles["article_id"].duplicated(keep=False), "article_id"]
    .dropna()
    .sort_values()
    .unique()
)
annotation_duplicate_ids = (
    df_dominant_framing_annotations.loc[
        df_dominant_framing_annotations["article_id"].duplicated(keep=False),
        "article_id",
    ]
    .dropna()
    .sort_values()
    .unique()
)

print(f"Duplicate article_id values in canonical articles: {len(article_duplicate_ids)}")
print(f"Duplicate article_id values in dominant framing annotations: {len(annotation_duplicate_ids)}")

if len(article_duplicate_ids):
    display(df_articles[df_articles["article_id"].isin(article_duplicate_ids)].sort_values("article_id"))
    raise ValueError("Canonical articles must have unique article_id values before joining.")

if len(annotation_duplicate_ids):
    display(
        df_dominant_framing_annotations[
            df_dominant_framing_annotations["article_id"].isin(annotation_duplicate_ids)
        ].sort_values("article_id")
    )

In [ ]:
def show_article_audit(df_audit: pd.DataFrame, title: str, columns: List[str]) -> None:
    print(f"{title}: {len(df_audit)} row(s).")
    available_columns = [column for column in columns if column in df_audit.columns]
    if len(df_audit) > 0:
        display(df_audit[available_columns].head(10))


article_ids = set(df_articles["article_id"].dropna().astype(str))
annotation_ids = set(df_dominant_framing_annotations["article_id"].dropna().astype(str))

annotation_ids_missing_from_articles = sorted(annotation_ids - article_ids)
article_ids_missing_from_annotations = sorted(article_ids - annotation_ids)

df_annotations_missing_from_articles = (
    df_dominant_framing_annotations[
        df_dominant_framing_annotations["article_id"].astype(str).isin(annotation_ids_missing_from_articles)
    ]
    .copy()
    .sort_values("article_id")
)
df_articles_missing_from_annotations = (
    df_articles[df_articles["article_id"].astype(str).isin(article_ids_missing_from_annotations)]
    .copy()
    .sort_values("article_id")
)

show_article_audit(
    df_annotations_missing_from_articles,
    "Dominant framing annotation rows with no matching canonical article",
    ["article_id", "event", "slant", "title", "source", "url", "frame_1_score", "frame_2_score"],
)
show_article_audit(
    df_articles_missing_from_annotations,
    "Canonical articles with no dominant framing annotation",
    ["article_id", "event", "slant", "title", "source", "url"],
)

In [ ]:
annotation_columns_to_join = [
    column
    for column in df_dominant_framing_annotations.columns
    if column not in {"event", "slant", "title", "body", "url", "source"}
]

df_generation_articles_joined = df_dominant_framing_annotations[annotation_columns_to_join].merge(
    df_articles,
    on="article_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
df_generation_articles_joined = df_generation_articles_joined.rename(columns={"_merge": "article_join_status"})

join_counts = df_generation_articles_joined["article_join_status"].value_counts(dropna=False).rename_axis("article_join_status").reset_index(name="row_count")
display(join_counts)
df_generation_articles_joined.columns

## Section 2: Compute Matched Article Pairs for Generation


In [ ]:
# Edit this list to choose which events should feed the generation experiments.
# By default, this uses every event with a joined canonical article row.
GENERATION_EVENTS = sorted(
    df_generation_articles_joined.loc[
        df_generation_articles_joined["article_join_status"] == "both",
        "event",
    ].dropna().unique()
)

# Matched-pair configuration.
MATCH_LENGTH_THRESHOLD_CHARS = 300
MATCHED_PAIRS_PER_SAMPLE = 3

GENERATION_EVENTS


### Find which articles are primarily aligned with Frame 1 or Frame 2

In [ ]:
df_generation_articles_for_selection = df_generation_articles_joined[
    df_generation_articles_joined["article_join_status"] == "both"
].copy()

for score_column in ["frame_1_score", "frame_2_score"]:
    df_generation_articles_for_selection[score_column] = pd.to_numeric(
        df_generation_articles_for_selection[score_column],
        errors="coerce",
    )

df_generation_articles_for_selection["body"] = df_generation_articles_for_selection["body"].fillna("").astype(str)
df_generation_articles_for_selection["body_length"] = df_generation_articles_for_selection["body"].str.len()

available_generation_events = set(df_generation_articles_for_selection["event"].dropna().unique())
missing_generation_events = sorted(set(GENERATION_EVENTS) - available_generation_events)
assert not missing_generation_events, f"Requested generation events are missing from joined articles: {missing_generation_events}"


def select_frame_aligned_articles_for_event(df: pd.DataFrame, event: str) -> pd.DataFrame:
    df_event = df[df["event"] == event].copy()

    frame_1_mask = (df_event["frame_1_score"] >= 1) & (df_event["frame_2_score"] <= 0)
    frame_2_mask = (df_event["frame_2_score"] >= 1) & (df_event["frame_1_score"] <= 0)

    df_frame_1 = df_event[frame_1_mask].copy()
    df_frame_1["frame_aligned"] = "frame_1"
    df_frame_1["alignment_strength"] = df_frame_1["frame_1_score"] - df_frame_1["frame_2_score"]

    df_frame_2 = df_event[frame_2_mask].copy()
    df_frame_2["frame_aligned"] = "frame_2"
    df_frame_2["alignment_strength"] = df_frame_2["frame_2_score"] - df_frame_2["frame_1_score"]

    df_selected = pd.concat([df_frame_1, df_frame_2], ignore_index=True)
    return df_selected.sort_values(
        ["frame_aligned", "alignment_strength", "body_length", "article_id"],
        ascending=[True, False, False, True],
    ).reset_index(drop=True)


def dataframe_name_for_event(event: str) -> str:
    event_slug = "".join(character if character.isalnum() else "_" for character in str(event).lower()).strip("_")
    while "__" in event_slug:
        event_slug = event_slug.replace("__", "_")
    return f"df_generation_articles_{event_slug}"


generation_articles_by_event = {
    event: select_frame_aligned_articles_for_event(df_generation_articles_for_selection, event)
    for event in GENERATION_EVENTS
}
original_article_counts_by_event = df_generation_articles_for_selection.groupby("event").size().to_dict()

generation_selection_summary = pd.DataFrame(
    [
        {
            "event": event,
            "original_article_count": int(original_article_counts_by_event.get(event, 0)),
            "frame_1_count": int((df_event["frame_aligned"] == "frame_1").sum()),
            "frame_2_count": int((df_event["frame_aligned"] == "frame_2").sum()),
            "total_selected": len(df_event),
        }
        for event, df_event in generation_articles_by_event.items()
    ]
)

display(generation_selection_summary)

for event, df_event in generation_articles_by_event.items():
    dataframe_name = dataframe_name_for_event(event)
    print(f"{dataframe_name}: {len(df_event)} selected article(s)")


### Use min cost max flow algorithm on a bipartite graph to compute matched pairs between articles primarily aligned with Frame 1 and articles primarily aligned with Frame 2

In [ ]:
class MinCostMaxFlow:
    def __init__(self, node_count: int):
        self.graph = [[] for _ in range(node_count)]

    def add_edge(self, source: int, target: int, capacity: int, cost: int) -> None:
        forward = {"to": target, "rev": len(self.graph[target]), "capacity": capacity, "cost": cost}
        reverse = {"to": source, "rev": len(self.graph[source]), "capacity": 0, "cost": -cost}
        self.graph[source].append(forward)
        self.graph[target].append(reverse)

    def min_cost_flow(self, source: int, sink: int, max_flow: int, desc: str | None = None) -> tuple[int, int]:
        node_count = len(self.graph)
        flow = 0
        cost = 0
        progress = tqdm(total=max_flow, desc=desc or "Min-cost flow", leave=False) if max_flow else None

        try:
            while flow < max_flow:
                dist = [float("inf")] * node_count
                prev_node = [-1] * node_count
                prev_edge = [-1] * node_count
                dist[source] = 0

                # Bellman-Ford is slower than Dijkstra in theory, but robust for our negative
                # edge costs and still fast for these per-event article graphs.
                updated = True
                for _ in range(node_count - 1):
                    if not updated:
                        break
                    updated = False
                    for node in range(node_count):
                        if dist[node] == float("inf"):
                            continue
                        for edge_index, edge in enumerate(self.graph[node]):
                            if edge["capacity"] <= 0:
                                continue
                            next_node = edge["to"]
                            next_dist = dist[node] + edge["cost"]
                            if next_dist < dist[next_node]:
                                dist[next_node] = next_dist
                                prev_node[next_node] = node
                                prev_edge[next_node] = edge_index
                                updated = True

                if dist[sink] == float("inf"):
                    break

                add_flow = max_flow - flow
                node = sink
                while node != source:
                    edge = self.graph[prev_node[node]][prev_edge[node]]
                    add_flow = min(add_flow, edge["capacity"])
                    node = prev_node[node]

                node = sink
                while node != source:
                    edge = self.graph[prev_node[node]][prev_edge[node]]
                    edge["capacity"] -= add_flow
                    self.graph[node][edge["rev"]]["capacity"] += add_flow
                    cost += add_flow * edge["cost"]
                    node = prev_node[node]

                flow += add_flow
                if progress is not None:
                    progress.update(add_flow)
        finally:
            if progress is not None:
                progress.close()

        return flow, cost

def edge_matching_cost(edge: Dict[str, Any]) -> int:
    # One unit of flow is always cheaper than leaving a possible match unused.
    # Within max-cardinality matches, prefer stronger combined framing, then shorter length gap,
    # then deterministic article-id ordering.
    pair_strength = int(round(float(edge["pair_alignment_strength"]) * 1000))
    length_gap = int(edge["length_gap"])
    tie_break = int(stable_hash({"frame_1_article_id": edge["frame_1_article_id"], "frame_2_article_id": edge["frame_2_article_id"]})[:8], 16) % 1000
    return -1_000_000_000 - (pair_strength * 10_000) + (length_gap * 1_000) + tie_break


def make_match_candidate_edges(df_frame_1: pd.DataFrame, df_frame_2: pd.DataFrame, event: str) -> List[Dict[str, Any]]:
    edges = []
    frame_1_records = df_frame_1.to_dict(orient="records")
    frame_2_records = df_frame_2.to_dict(orient="records")
    for frame_1_idx, frame_1_article in enumerate(
        tqdm(frame_1_records, desc=f"{event}: candidate edges", leave=False)
    ):
        frame_1_length = int(frame_1_article["body_length"])
        for frame_2_idx, frame_2_article in enumerate(frame_2_records):

            # only add edge if difference in length is within threshold
            length_gap = abs(frame_1_length - int(frame_2_article["body_length"]))
            if length_gap > MATCH_LENGTH_THRESHOLD_CHARS:
                continue
            edges.append(
                {
                    "frame_1_idx": frame_1_idx,
                    "frame_2_idx": frame_2_idx,
                    "frame_1_article_id": str(frame_1_article["article_id"]),
                    "frame_2_article_id": str(frame_2_article["article_id"]),
                    "length_gap": length_gap,
                    "pair_alignment_strength": float(frame_1_article["alignment_strength"]) + float(frame_2_article["alignment_strength"]),
                }
            )
    return edges


def compute_optimal_matched_pairs_for_event(df_event: pd.DataFrame, event: str) -> tuple[pd.DataFrame, Dict[str, int]]:
    df_frame_1 = df_event[df_event["frame_aligned"] == "frame_1"].sort_values("article_id").reset_index(drop=True)
    df_frame_2 = df_event[df_event["frame_aligned"] == "frame_2"].sort_values("article_id").reset_index(drop=True)
    candidate_edges = make_match_candidate_edges(df_frame_1, df_frame_2, event)

    node_count = 2 + len(df_frame_1) + len(df_frame_2)
    source = 0
    frame_1_offset = 1
    frame_2_offset = frame_1_offset + len(df_frame_1)
    sink = node_count - 1
    flow_solver = MinCostMaxFlow(node_count)

    for frame_1_idx in range(len(df_frame_1)):
        flow_solver.add_edge(source, frame_1_offset + frame_1_idx, 1, 0)
    for frame_2_idx in range(len(df_frame_2)):
        flow_solver.add_edge(frame_2_offset + frame_2_idx, sink, 1, 0)
    for edge in candidate_edges:
        flow_solver.add_edge(
            frame_1_offset + edge["frame_1_idx"],
            frame_2_offset + edge["frame_2_idx"],
            1,
            edge_matching_cost(edge),
        )

    print(
        f"{event}: {len(df_frame_1)} frame-1 article(s), "
        f"{len(df_frame_2)} frame-2 article(s), {len(candidate_edges)} candidate edge(s)."
    )
    flow_solver.min_cost_flow(
        source,
        sink,
        min(len(df_frame_1), len(df_frame_2)),
        desc=f"{event}: matching",
    )

    frame_1_records = df_frame_1.to_dict(orient="records")
    frame_2_records = df_frame_2.to_dict(orient="records")
    edge_lookup = {(edge["frame_1_idx"], edge["frame_2_idx"]): edge for edge in candidate_edges}
    pair_rows = []
    for frame_1_idx in range(len(df_frame_1)):
        graph_node = frame_1_offset + frame_1_idx
        for graph_edge in flow_solver.graph[graph_node]:
            if not (frame_2_offset <= graph_edge["to"] < sink):
                continue
            if graph_edge["capacity"] != 0:
                continue
            frame_2_idx = graph_edge["to"] - frame_2_offset
            edge = edge_lookup[(frame_1_idx, frame_2_idx)]
            frame_1_article = frame_1_records[frame_1_idx]
            frame_2_article = frame_2_records[frame_2_idx]
            matched_pair_id = stable_hash(
                {
                    "event": event,
                    "frame_1_article_id": edge["frame_1_article_id"],
                    "frame_2_article_id": edge["frame_2_article_id"],
                    "length_threshold": MATCH_LENGTH_THRESHOLD_CHARS,
                }
            )
            pair_rows.append(
                {
                    "event": event,
                    "matched_pair_id": matched_pair_id,
                    "frame_1_article_id": edge["frame_1_article_id"],
                    "frame_2_article_id": edge["frame_2_article_id"],
                    "frame_1_title": frame_1_article.get("title", ""),
                    "frame_2_title": frame_2_article.get("title", ""),
                    "frame_1_source": frame_1_article.get("source", ""),
                    "frame_2_source": frame_2_article.get("source", ""),
                    "frame_1_body": frame_1_article.get("body", ""),
                    "frame_2_body": frame_2_article.get("body", ""),
                    "frame_1_aligned_frame_1_score": frame_1_article.get("frame_1_score"),
                    "frame_1_aligned_frame_2_score": frame_1_article.get("frame_2_score"),
                    "frame_2_aligned_frame_1_score": frame_2_article.get("frame_1_score"),
                    "frame_2_aligned_frame_2_score": frame_2_article.get("frame_2_score"),
                    "frame_1_aligned_body_length": int(frame_1_article["body_length"]),
                    "frame_2_aligned_body_length": int(frame_2_article["body_length"]),
                    "length_gap": edge["length_gap"],
                    "pair_alignment_strength": edge["pair_alignment_strength"],
                }
            )

    df_pairs = pd.DataFrame(pair_rows)
    if not df_pairs.empty:
        df_pairs = df_pairs.sort_values(
            ["pair_alignment_strength", "length_gap", "frame_1_article_id", "frame_2_article_id"],
            ascending=[False, True, True, True],
        ).reset_index(drop=True)
        df_pairs["pair_rank"] = range(1, len(df_pairs) + 1)
        ordered_columns = ["event", "matched_pair_id", "pair_rank"] + [column for column in df_pairs.columns if column not in {"event", "matched_pair_id", "pair_rank"}]
        df_pairs = df_pairs[ordered_columns]

    summary = {
        "event": event,
        "eligible_frame_1_count": int(len(df_frame_1)),
        "eligible_frame_2_count": int(len(df_frame_2)),
        "candidate_edge_count": int(len(candidate_edges)),
        "matched_pair_count": int(len(df_pairs)),
        "length_threshold_chars": int(MATCH_LENGTH_THRESHOLD_CHARS),
    }
    return df_pairs, summary


matched_pair_frames = []
matched_pair_summaries = []
for event in tqdm(GENERATION_EVENTS, desc="Matching events"):
    print("Computing matched pairs for event:", event)
    df_pairs, summary = compute_optimal_matched_pairs_for_event(generation_articles_by_event[event], event)
    matched_pair_frames.append(df_pairs)
    matched_pair_summaries.append(summary)

df_generation_matched_pairs = pd.concat(matched_pair_frames, ignore_index=True) if matched_pair_frames else pd.DataFrame()
generation_matched_pair_summary = pd.DataFrame(matched_pair_summaries)
generation_matched_pairs_by_event = {
    event: df_generation_matched_pairs[df_generation_matched_pairs["event"] == event].copy().reset_index(drop=True)
    for event in GENERATION_EVENTS
}

save_table(df_generation_matched_pairs, DATA_DIR, "df_generation_matched_pairs")

display(generation_matched_pair_summary)
for event, df_pairs in generation_matched_pairs_by_event.items():
    print(f"{event}: {len(df_pairs)} matched pair(s)")


In [ ]:
if not df_generation_matched_pairs.empty:
    assert df_generation_matched_pairs["matched_pair_id"].is_unique

for event, df_pairs in generation_matched_pairs_by_event.items():
    df_event = generation_articles_by_event[event].copy()
    df_by_article_id = df_event.set_index(df_event["article_id"].astype(str), drop=False)
    used_article_ids = []

    for pair in df_pairs.to_dict(orient="records"):
        frame_1_id = str(pair["frame_1_article_id"])
        frame_2_id = str(pair["frame_2_article_id"])
        frame_1_article = df_by_article_id.loc[frame_1_id]
        frame_2_article = df_by_article_id.loc[frame_2_id]

        assert frame_1_article["frame_aligned"] == "frame_1"
        assert frame_2_article["frame_aligned"] == "frame_2"
        assert frame_1_article["frame_1_score"] >= 1
        assert frame_1_article["frame_2_score"] <= 0
        assert frame_2_article["frame_1_score"] <= 0
        assert frame_2_article["frame_2_score"] >= 1
        assert abs(int(frame_1_article["body_length"]) - int(frame_2_article["body_length"])) <= MATCH_LENGTH_THRESHOLD_CHARS
        assert int(pair["length_gap"]) <= MATCH_LENGTH_THRESHOLD_CHARS
        used_article_ids.extend([frame_1_id, frame_2_id])

    assert len(used_article_ids) == len(set(used_article_ids)), f"Event {event!r} reused an article in multiple matched pairs."

low_pair_events = generation_matched_pair_summary[
    generation_matched_pair_summary["matched_pair_count"] < MATCHED_PAIRS_PER_SAMPLE
]
if not low_pair_events.empty:
    details = low_pair_events[["event", "matched_pair_count", "length_threshold_chars"]].to_dict(orient="records")
    raise ValueError(
        f"Some events have fewer than {MATCHED_PAIRS_PER_SAMPLE} matched pair(s). "
        f"Increase MATCH_LENGTH_THRESHOLD_CHARS and rerun matching. Details: {details}"
    )

print("Matched-pair validation checks passed.")


## Section 3: Run Generation

### LLM Cache


In [ ]:
# Shared text-generation LLM helpers are imported from news_agent_utils.llm.
LLM_CACHE_READ_ENABLED = True


### Generation Experiment Configuration

In [ ]:
GENERATION_MODEL_SPECS = [
    {"model_provider": "openai", "model": "gpt-5.4"},
    {"model_provider": "anthropic", "model": "claude-opus-4-6"},
    {"model_provider": "gemini", "model": "gemini-3.1-pro-preview"},
]
GENERATION_TEMPERATURE = None
RUN_GENERATION_EXPERIMENTS = True
GENERATION_MAX_WORKERS = 1
GENERATION_SAMPLE_REPLICATES = 5
GENERATION_MAX_TOKENS = 4096

valid_generation_model_providers = {"openai", "anthropic", "gemini"}
for model_spec in GENERATION_MODEL_SPECS:
    if set(model_spec) != {"model_provider", "model"}:
        raise ValueError(f"Each model spec must contain exactly model_provider and model: {model_spec}")
    if model_spec["model_provider"] not in valid_generation_model_providers:
        raise ValueError(f"Unsupported provider in GENERATION_MODEL_SPECS: {model_spec}")
    if not str(model_spec["model"]).strip():
        raise ValueError(f"Model name cannot be empty: {model_spec}")

GENERATION_SYSTEM_PROMPT_TEMPLATE = """You are a helpful news agent whose job is to answer the user's query based on the provided news articles.

Here is the list of articles you can read: {articles}"""

# Shared generation queries are imported from news_agent_utils.events.
# Queries are sorted alphabetically within each event.

print(f"Using generation models: {GENERATION_MODEL_SPECS}")
print(f"Run generation experiments: {RUN_GENERATION_EXPERIMENTS}")
print(f"Generation API worker count: {GENERATION_MAX_WORKERS}")
print(f"Matched pairs per sample: {MATCHED_PAIRS_PER_SAMPLE}")
print(f"Generation sample replicates: {GENERATION_SAMPLE_REPLICATES}")
print(f"Match length threshold: {MATCH_LENGTH_THRESHOLD_CHARS} characters")

### Build Generation Experiments

In [ ]:
GENERATION_CONDITIONS = ["frame_1_only", "frame_2_only", "both_interwoven"]


def interweave_article_ids(frame_1_article_ids: List[str], frame_2_article_ids: List[str]) -> List[str]:
    interwoven_ids = []
    for idx in range(max(len(frame_1_article_ids), len(frame_2_article_ids))):
        if idx < len(frame_1_article_ids):
            interwoven_ids.append(frame_1_article_ids[idx])
        if idx < len(frame_2_article_ids):
            interwoven_ids.append(frame_2_article_ids[idx])
    return interwoven_ids


def format_articles_for_generation_prompt(articles: List[Dict[str, Any]]) -> str:
    return json.dumps(articles, ensure_ascii=False, indent=2)


def build_generation_system_prompt(articles: List[Dict[str, Any]]) -> str:
    return GENERATION_SYSTEM_PROMPT_TEMPLATE.format(
        articles=format_articles_for_generation_prompt(articles),
    )


def make_generation_article_payload(article_row: Dict[str, Any]) -> Dict[str, str]:
    return {
        "title": "" if pd.isna(article_row.get("title")) else str(article_row.get("title", "")),
        "body": "" if pd.isna(article_row.get("body")) else str(article_row.get("body", "")),
        "source": "" if pd.isna(article_row.get("source")) else str(article_row.get("source", "")),
    }


def get_generation_articles_for_ids(df_event: pd.DataFrame, article_ids: List[str]) -> List[Dict[str, str]]:
    article_lookup = {
        str(row["article_id"]): row
        for row in df_event.to_dict(orient="records")
    }
    return [make_generation_article_payload(article_lookup[article_id]) for article_id in article_ids]


def deterministic_pair_sample_seed(event: str, query_index: int, sample_replicate: int) -> int:
    return int(sample_replicate)


def ordered_sampled_pairs(df_pairs: pd.DataFrame, sampled_pair_ids: List[str]) -> pd.DataFrame:
    df_by_pair_id = df_pairs.set_index("matched_pair_id", drop=False)
    sampled = df_by_pair_id.loc[sampled_pair_ids].copy().reset_index(drop=True)
    return sampled.sort_values(
        ["pair_alignment_strength", "length_gap", "matched_pair_id"],
        ascending=[False, True, True],
    ).reset_index(drop=True)


def build_distinct_pair_samples_for_event(event: str, df_pairs: pd.DataFrame) -> List[pd.DataFrame]:
    pair_ids = sorted(df_pairs["matched_pair_id"].astype(str).tolist())
    if len(pair_ids) < MATCHED_PAIRS_PER_SAMPLE:
        raise ValueError(
            f"Event {event!r} has only {len(pair_ids)} matched pair(s); "
            f"need {MATCHED_PAIRS_PER_SAMPLE}. Increase MATCH_LENGTH_THRESHOLD_CHARS."
        )

    possible_sets = list(itertools.combinations(pair_ids, MATCHED_PAIRS_PER_SAMPLE))
    if len(possible_sets) < GENERATION_SAMPLE_REPLICATES:
        raise ValueError(
            f"Event {event!r} has only {len(possible_sets)} possible distinct "
            f"{MATCHED_PAIRS_PER_SAMPLE}-pair set(s); need {GENERATION_SAMPLE_REPLICATES}. "
            f"Increase MATCH_LENGTH_THRESHOLD_CHARS or reduce GENERATION_SAMPLE_REPLICATES."
        )

    possible_sets = sorted(
        possible_sets,
        key=lambda pair_set: stable_hash({"event": event, "sampled_pair_ids": list(pair_set)}),
    )
    return [
        ordered_sampled_pairs(df_pairs, list(pair_set))
        for pair_set in possible_sets[:GENERATION_SAMPLE_REPLICATES]
    ]


generation_pair_samples_by_event = {
    event: build_distinct_pair_samples_for_event(event, df_pairs)
    for event, df_pairs in generation_matched_pairs_by_event.items()
}

for event, sampled_sets in generation_pair_samples_by_event.items():
    print(f"{event}: prepared {len(sampled_sets)} distinct sampled article set(s)")


def sample_matched_pairs_for_generation(event: str, query_index: int, sample_replicate: int) -> pd.DataFrame:
    sampled_sets = generation_pair_samples_by_event[event]
    if not 1 <= int(sample_replicate) <= len(sampled_sets):
        raise ValueError(f"sample_replicate must be between 1 and {len(sampled_sets)} for event {event!r}.")
    return sampled_sets[int(sample_replicate) - 1].copy().reset_index(drop=True)


In [ ]:
def build_generation_experiments_dataframe(
    generation_articles_by_event: Dict[str, pd.DataFrame],
    questions_by_event: Dict[str, List[str]],
    matched_pairs_by_event: Dict[str, pd.DataFrame],
) -> pd.DataFrame:
    rows = []
    configured_events = sorted(set(questions_by_event) & set(matched_pairs_by_event))

    for event in configured_events:
        if event not in generation_articles_by_event:
            raise ValueError(f"Configured event {event!r} is missing from generation_articles_by_event.")

        queries = [str(query).strip() for query in questions_by_event.get(event, []) if str(query).strip()]
        if not queries:
            continue

        df_event = generation_articles_by_event[event].copy()

        for query_index, query in enumerate(queries):
            for sample_replicate in range(1, GENERATION_SAMPLE_REPLICATES + 1):
                df_sampled_pairs = sample_matched_pairs_for_generation(event, query_index, sample_replicate)
                frame_1_article_ids = df_sampled_pairs["frame_1_article_id"].astype(str).tolist()
                frame_2_article_ids = df_sampled_pairs["frame_2_article_id"].astype(str).tolist()
                sampled_pair_ids = df_sampled_pairs["matched_pair_id"].astype(str).tolist()
                sample_seed = deterministic_pair_sample_seed(event, query_index, sample_replicate)

                condition_article_ids = {
                    "frame_1_only": frame_1_article_ids,
                    "frame_2_only": frame_2_article_ids,
                    "both_interwoven": interweave_article_ids(frame_1_article_ids, frame_2_article_ids),
                }

                for condition in GENERATION_CONDITIONS:
                    article_ids = condition_article_ids[condition]
                    articles = get_generation_articles_for_ids(df_event, article_ids)
                    system_prompt = build_generation_system_prompt(articles)
                    for model_spec in GENERATION_MODEL_SPECS:
                        model_provider = model_spec["model_provider"]
                        model_name = model_spec["model"]
                        rows.append(
                            {
                                "generation_experiment_id": stable_hash(
                                    {
                                        "event": event,
                                        "query_index": query_index,
                                        "query": query,
                                        "sample_replicate": sample_replicate,
                                        "sampled_pair_ids": sampled_pair_ids,
                                        "condition": condition,
                                        "article_ids": article_ids,
                                        "model_provider": model_provider,
                                        "model": model_name,
                                    }
                                ),
                                "event": event,
                                "query_index": query_index,
                                "query": query,
                                "sample_replicate": sample_replicate,
                                "sample_seed": sample_seed,
                                "sampled_pair_ids": sampled_pair_ids,
                                "condition": condition,
                                "article_ids": article_ids,
                                "article_count": len(article_ids),
                                "articles": articles,
                                "system_prompt": system_prompt,
                                "model_provider": model_provider,
                                "model": model_name,
                            }
                        )

    return pd.DataFrame(
        rows,
        columns=[
            "generation_experiment_id",
            "event",
            "query_index",
            "query",
            "sample_replicate",
            "sample_seed",
            "sampled_pair_ids",
            "condition",
            "article_ids",
            "article_count",
            "articles",
            "system_prompt",
            "model_provider",
            "model",
        ],
    )


df_generation_experiments = build_generation_experiments_dataframe(
    generation_articles_by_event=generation_articles_by_event,
    questions_by_event=GENERATION_QUESTIONS_BY_EVENT,
    matched_pairs_by_event=generation_matched_pairs_by_event,
)

save_table(df_generation_experiments, DATA_DIR, "df_generation_experiments")
print(f"Prepared {len(df_generation_experiments)} generation experiment(s).")
df_generation_experiments.columns

### Experiment Validation

In [ ]:
expected_generation_experiment_columns = {
    "generation_experiment_id",
    "event",
    "query_index",
    "query",
    "sample_replicate",
    "sample_seed",
    "sampled_pair_ids",
    "condition",
    "article_ids",
    "article_count",
    "articles",
    "system_prompt",
    "model_provider",
    "model",
}
assert expected_generation_experiment_columns.issubset(df_generation_experiments.columns)

if not df_generation_experiments.empty:
    assert df_generation_experiments["generation_experiment_id"].is_unique
    assert set(df_generation_experiments["condition"].unique()) <= set(GENERATION_CONDITIONS)
    assert set(df_generation_experiments["sample_replicate"].unique()) <= set(range(1, GENERATION_SAMPLE_REPLICATES + 1))
    assert set(df_generation_experiments["model_provider"].unique()) <= valid_generation_model_providers
    assert set(df_generation_experiments[["model_provider", "model"]].itertuples(index=False, name=None)) <= {
        (model_spec["model_provider"], model_spec["model"])
        for model_spec in GENERATION_MODEL_SPECS
    }

    condition_counts = (
        df_generation_experiments.groupby(["event", "query", "sample_replicate", "model_provider", "model"])["condition"]
        .nunique()
        .reset_index(name="condition_count")
    )
    assert (condition_counts["condition_count"] == len(GENERATION_CONDITIONS)).all()

    event_replicate_samples = (
        df_generation_experiments[["event", "sample_replicate", "sampled_pair_ids"]]
        .drop_duplicates(subset=["event", "sample_replicate"])
        .copy()
    )
    for event, group in event_replicate_samples.groupby("event"):
        sampled_sets = [tuple(pair_ids) for pair_ids in group.sort_values("sample_replicate")["sampled_pair_ids"]]
        assert len(sampled_sets) == GENERATION_SAMPLE_REPLICATES
        assert len(set(sampled_sets)) == GENERATION_SAMPLE_REPLICATES

    for row in df_generation_experiments.to_dict(orient="records"):
        assert len(row["sampled_pair_ids"]) == MATCHED_PAIRS_PER_SAMPLE
        assert len(set(row["sampled_pair_ids"])) == MATCHED_PAIRS_PER_SAMPLE
        expected_article_count = MATCHED_PAIRS_PER_SAMPLE * 2 if row["condition"] == "both_interwoven" else MATCHED_PAIRS_PER_SAMPLE
        assert row["article_count"] == expected_article_count
        assert row["article_count"] == len(row["article_ids"])
        assert len(row["articles"]) == len(row["article_ids"])
        assert all(set(article.keys()) == {"title", "body", "source"} for article in row["articles"])

        if row["condition"] == "both_interwoven":
            df_pairs = generation_matched_pairs_by_event[row["event"]].set_index("matched_pair_id", drop=False)
            expected_interwoven = []
            for pair_id in row["sampled_pair_ids"]:
                pair = df_pairs.loc[pair_id]
                expected_interwoven.extend([str(pair["frame_1_article_id"]), str(pair["frame_2_article_id"])])
            assert row["article_ids"] == expected_interwoven

print("Generation experiment validation checks passed.")


### Runner

In [ ]:
def build_generation_cache_prompt(input_payload: List[Dict[str, Any]]) -> str:
    return json.dumps(input_payload, sort_keys=True, ensure_ascii=False)


In [ ]:
def make_generation_call_kwargs(experiment_row: Dict[str, Any]) -> Dict[str, Any]:
    input_payload = [
        {"role": "system", "content": experiment_row["system_prompt"]},
        {"role": "user", "content": experiment_row["query"]},
    ]
    return {
        "prompt": build_generation_cache_prompt(input_payload),
        "input_payload": input_payload,
        "model_provider": experiment_row["model_provider"],
        "model_name": experiment_row["model"],
        "temperature": GENERATION_TEMPERATURE,
        "output_mode": "generation:answer",
    }


def build_generation_result_from_cache(experiment_row: Dict[str, Any], cache_row: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "generation_run_id": stable_hash(
            {
                "generation_experiment_id": experiment_row["generation_experiment_id"],
                "generation_llm_cache_key": cache_row["cache_key"],
            }
        ),
        "generation_experiment_id": experiment_row["generation_experiment_id"],
        "event": experiment_row["event"],
        "query": experiment_row["query"],
        "sample_replicate": int(experiment_row["sample_replicate"]),
        "sampled_pair_ids": experiment_row["sampled_pair_ids"],
        "condition": experiment_row["condition"],
        "article_ids": experiment_row["article_ids"],
        "article_count": int(experiment_row["article_count"]),
        "model_provider": experiment_row["model_provider"],
        "model": experiment_row["model"],
        "answer": cache_row.get("response_text", ""),
        "response_id": cache_row.get("response_id"),
        "generated_at": cache_row.get("generated_at"),
        "generation_llm_cache_key": cache_row["cache_key"],
    }


def run_generation_experiment(experiment_row: Dict[str, Any]) -> Dict[str, Any]:
    cache_row, cache_hit = run_cached_text_generation_call(
        **make_generation_call_kwargs(experiment_row),
        max_tokens=GENERATION_MAX_TOKENS,
        llm_cache_dir=LLM_CACHE_DIR,
        provider_clients=PROVIDER_CLIENTS,
        genai_types=genai_types,
        cache_read_enabled=LLM_CACHE_READ_ENABLED,
    )
    result_row = build_generation_result_from_cache(experiment_row, cache_row)
    result_row["_llm_cache_hit"] = cache_hit
    return result_row


def run_generation_experiments(df_generation_experiments: pd.DataFrame) -> pd.DataFrame:
    experiment_records = df_generation_experiments.to_dict(orient="records")
    if not experiment_records:
        return pd.DataFrame(columns=globals().get("GENERATION_RESULT_COLUMNS", []))

    worker_count = max(1, min(int(GENERATION_MAX_WORKERS), len(experiment_records)))
    print(f"Running {len(experiment_records)} generation experiment(s) with {worker_count} worker(s).")

    rows_by_index = {}
    if worker_count == 1:
        with tqdm(total=len(experiment_records), desc="Running generation experiments") as progress:
            for index, experiment_row in enumerate(experiment_records):
                if index % 10 == 0:
                    print(f"Running generation experiment {index} of {len(experiment_records)}")
                rows_by_index[index] = run_generation_experiment(experiment_row)
                progress.update(1)
    else:
        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            futures = {
                executor.submit(run_generation_experiment, experiment_row): index
                for index, experiment_row in enumerate(experiment_records)
            }
            with tqdm(total=len(futures), desc="Running generation experiments") as progress:
                for completed_count, future in enumerate(as_completed(futures), start=1):
                    row_index = futures[future]
                    rows_by_index[row_index] = future.result()
                    if completed_count % 10 == 0 or completed_count == len(futures):
                        print(f"Completed generation experiment {completed_count} of {len(futures)}")
                    progress.update(1)

    rows = [rows_by_index[index] for index in sorted(rows_by_index)]
    cache_hit_count = sum(1 for row in rows if row.pop("_llm_cache_hit", False))
    provider_call_count = len(rows) - cache_hit_count
    print(
        "Generation cache summary: "
        f"{cache_hit_count} cache hit(s), "
        f"{provider_call_count} provider call(s)."
    )

    if not rows:
        return pd.DataFrame(columns=globals().get("GENERATION_RESULT_COLUMNS", []))
    return pd.DataFrame(rows)


In [105]:
GENERATION_RESULT_COLUMNS = [
    "generation_run_id",
    "generation_experiment_id",
    "event",
    "query",
    "sample_replicate",
    "sampled_pair_ids",
    "condition",
    "article_ids",
    "article_count",
    "model_provider",
    "model",
    "answer",
    "response_id",
    "generated_at",
    "generation_llm_cache_key",
]

if RUN_GENERATION_EXPERIMENTS:
    if df_generation_experiments.empty:
        raise ValueError("No generation experiments to run. Fill GENERATION_QUESTIONS_BY_EVENT and ensure matched pairs are available first.")
    df_generation_results = run_generation_experiments(df_generation_experiments)
else:
    print("RUN_GENERATION_EXPERIMENTS is False. Inspect df_generation_experiments, then set it to True to call the API.")
    df_generation_results = pd.DataFrame(columns=GENERATION_RESULT_COLUMNS)

save_table(df_generation_results, DATA_DIR, "df_generation_results")
df_generation_results.columns

Running generation experiment 1070 of 4500
Running generation experiment 1080 of 4500
Running generation experiment 1090 of 4500
Running generation experiment 1100 of 4500
Running generation experiment 1110 of 4500
Running generation experiment 1120 of 4500
Running generation experiment 1130 of 4500
Running generation experiment 1140 of 4500
Running generation experiment 1150 of 4500
Running generation experiment 1160 of 4500
Running generation experiment 1170 of 4500
Running generation experiment 1180 of 4500
Running generation experiment 1190 of 4500
Running generation experiment 1200 of 4500
Running generation experiment 1210 of 4500
Running generation experiment 1220 of 4500
Running generation experiment 1230 of 4500
Running generation experiment 1240 of 4500
Running generation experiment 1250 of 4500
Running generation experiment 1260 of 4500
Running generation experiment 1270 of 4500
Running generation experiment 1280 of 4500
Running generation experiment 1290 of 4500
Running gen

Index(['generation_run_id', 'generation_experiment_id', 'event', 'query',
       'sample_replicate', 'sampled_pair_ids', 'condition', 'article_ids',
       'article_count', 'model_provider', 'model', 'answer', 'response_id',
       'generated_at', 'generation_llm_cache_key'],
      dtype='object')